# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Display dataset metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'Unknown')}")
print(f"Version: {getattr(metadata, 'version', 'Unknown')}")
print(f"License: {getattr(metadata, 'license', 'Unknown')}\n")
print("Keywords:", getattr(metadata, 'keywords', []))

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
from mlcroissant.dataset.metadata import utils

# Utility to extract '@id' fields from Croissant objects
def get_id(obj):
    if hasattr(obj, 'id'):
        return obj.id
    if hasattr(obj, '@id'):
        return getattr(obj, '@id')
    if isinstance(obj, dict) and '@id' in obj:
        return obj['@id']
    return str(obj)

# List all record sets and their IDs
record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    if isinstance(metadata.recordSet, list):
        record_sets = metadata.recordSet
    else:
        record_sets = [metadata.recordSet]
else:
    # Try parsing record sets from metadata JSON if not present as objects
    meta_j = dataset.metadata.to_json()
    if 'recordSet' in meta_j:
        record_sets = meta_j['recordSet'] if isinstance(meta_j['recordSet'], list) else [meta_j['recordSet']]

print("Record sets (@id):")
for rs in record_sets:
    rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else get_id(rs)
    print("-", rs_id)

# If there are no record sets, try to deduce them from the dataset
if not record_sets:
    print("No record sets found in the top-level metadata. Attempting to infer from Croissant schema...")
    # mlcroissant exposes record set ids
    try:
        available_record_sets = list(dataset.available_record_sets)
        if available_record_sets:
            print("Discovered record sets via mlcroissant:")
            for rsid in available_record_sets:
                print("-", rsid)
    except Exception as e:
        print("Could not infer record sets:", e)

# Let's use auto-detection for the remainder
record_sets_ids = []
try:
    record_sets_ids = list(dataset.available_record_sets)
except Exception as e:
    print("Could not list available record sets.", e)

# For each record set, print a sample record and available fields
print("\nSample record and fields for each record set:")
for record_set_id in record_sets_ids:
    print(f"\nRecord set '@id': {record_set_id}")
    records_iter = dataset.records(record_set=record_set_id)
    sample_record = None
    for rec in records_iter:
        sample_record = rec
        break
    if sample_record:
        print(f"  Fields/columns: {list(sample_record.keys())}")
    else:
        print("  [no records found]")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all available record sets by @id
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"---\nLoaded DataFrame for Record Set @id='{record_set_id}': shape {df.shape}")
    print("Columns:", df.columns.tolist()[:10], ('...' if len(df.columns) > 10 else ''))

# Select the main record set for downstream EDA (pick the largest frame)
main_record_set_id = max(dataframes, key=lambda k: dataframes[k].shape[0]) if dataframes else None
if main_record_set_id is not None:
    print(f"\nAnalyzing record set: {main_record_set_id}")
    print(dataframes[main_record_set_id].head())
else:
    print("No records available for analysis.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Automatic numeric field detection for demonstration
import numpy as np

df = dataframes[main_record_set_id].copy()

# Find a numeric field (float or int columns)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_cols:
    # Try to coerce columns to numeric
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except (ValueError, TypeError):
            continue
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if numeric_cols:
    numeric_field = numeric_cols[0]
    print(f"Using numeric field for demo: '{numeric_field}'")
else:
    print("No usable numeric field found.")
    numeric_field = None

if numeric_field:
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized '{numeric_field}' for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Attempt to find a group field (first string/categorical with low cardinality)
    group_field = None
    group_candidates = [col for col in df.columns if (df[col].dtype==object or df[col].dtype=='category')] 
    for col in group_candidates:
        if df[col].nunique() > 1 and df[col].nunique() < 10:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped data by '{group_field}', mean of '{numeric_field}':")
        print(grouped_df)
else:
    print("Skipping EDA: No numeric column in the extracted DataFrame.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

    # Boxplot by group_field if existed
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we demonstrated how to load and explore the FAIR² dataset, "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells", using the `mlcroissant` library.

- We accessed metadata and listed available record set `@id`s.
- We loaded records from a selected record set and performed exploratory data analysis, including filtering on a numeric field, normalization, grouping, and visualization.
- The dataset contains valuable quantitative data for proteomics and mass spectrometry studies on human extracellular vesicles.

For more customized analysis or to work with specific columns, refer to the Croissant schema documentation for the exact field and column `@id`s, and adjust the EDA section accordingly.